# 🧪 PyTorch Lab 10 — Explainable AI on a fine-tuned ResNet18

In this lab, you will build a **complete XAI workflow** for image classification.

Instead of training a CNN from scratch, you will:
1. use a **small natural image dataset**;
2. fine-tune a **pretrained ResNet18**;
3. explain predictions with **LIME**;
4. explain predictions with **Grad-CAM**.

## Learning goals
By the end of this practical session, you should be able to:
- load and preprocess a small natural image dataset (CIFAR-10) in PyTorch;
- fine-tune a pretrained CNN (ResNet18) for image classification;
- visualize some model predictions on test images;
- generate a **LIME** explanation for one prediction;
- generate a **Grad-CAM** explanation for one prediction;
- compare these two explanation methods and discuss their limits.

## Part 1 — Setup and reproducibility

In [ ]:
import os
import random
from pathlib import Path
from time import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

---

## Part 2 — Dataset and preprocessing

### Chosen dataset: CIFAR-10

We use **CIFAR-10** because:
- it is directly available in `torchvision`;
- it contains **10 classes** of natural color images;
- it is relatively small and well suited for a practical session;
- after resizing to `224 × 224`, it can be used with a pretrained **ResNet18**.

In [ ]:
data_dir = Path("./data")

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), # Convert values to [0, 1]
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

full_train_dataset = torchvision.datasets.CIFAR10(
    root=data_dir,
    train=True,
    download=True,
    transform=train_transform,
)

full_test_dataset = torchvision.datasets.CIFAR10(
    root=data_dir,
    train=False,
    download=True,
    transform=test_transform,
)

class_names = full_train_dataset.classes
print("Classes:", class_names)
print("Full training size:", len(full_train_dataset))
print("Full test size    :", len(full_test_dataset))

### Build smaller train / validation / test subsets

To keep training reasonably fast, we do **not** use the full training set.  Instead, we create a **smaller subset** of CIFAR-10.

Use the following sizes:
- `train_subset_size = 10000`
- `val_subset_size = 2000`
- `test_subset_size = 2000`

Then create:
- `train_loader`
- `val_loader`
- `test_loader`

Use:
- `batch_size = 64`
- shuffle only for the training loader

**Hint:** You can first select random indices, then wrap them with `Subset(...)`.

In [ ]:
train_subset_size = 10000
val_subset_size = 2000
test_subset_size = 2000

train_val_indices = torch.randperm(len(full_train_dataset), generator=torch.Generator().manual_seed(SEED)).tolist()
train_indices = train_val_indices[:train_subset_size]
val_indices = train_val_indices[train_subset_size:train_subset_size + val_subset_size]

test_indices = torch.randperm(len(full_test_dataset), generator=torch.Generator().manual_seed(SEED)).tolist()[:test_subset_size]

train_dataset = Subset(full_train_dataset, train_indices)
val_dataset = Subset(
    torchvision.datasets.CIFAR10(root=data_dir, train=True, download=False, transform=test_transform),
    val_indices
)
test_dataset = Subset(full_test_dataset, test_indices)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print("Train subset size:", len(train_dataset))
print("Val subset size  :", len(val_dataset))
print("Test subset size :", len(test_dataset))

### Visualize a mini-batch

Before fine-tuning the model, inspect a few images and labels.

In [ ]:
def denormalize(img, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return img * std + mean

images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    # Convert the normalized tensor back to a displayable image.
    # Change the shape from (C, H, W) to (H, W, C) and keep pixel values in [0, 1].
    img = denormalize(images[i], imagenet_mean, imagenet_std).permute(1, 2, 0).clamp(0, 1)
    plt.imshow(img)
    plt.title(class_names[labels[i].item()])
    plt.axis("off")
plt.tight_layout()
plt.show()

---

## Part 3 — Fine-tuning a pretrained ResNet18

We now use a **pretrained ResNet18** and adapt it to CIFAR-10.

### Strategy
1. load a pretrained `ResNet18`;
2. replace the final classification layer;
3. freeze most of the network;
4. fine-tune only `layer4` and `fc`.

This is enough for the XAI part of the lab, because we only need a reasonably good CNN to explain.

### Question before coding

Why is fine-tuning a pretrained network a good choice for this lab, compared with training a CNN from scratch?

### Expected answer

Fine-tuning is a good choice here because:
- pretrained filters already capture useful generic visual patterns;
- training becomes faster and more stable;
- we need less data than for training from scratch;
- the practical session can focus on **explainability** rather than on long training;
- a better model usually produces more meaningful explanations.

### Instantiate the model

Complete the next cell so that:
- the final layer outputs **10 logits**;
- all parameters are frozen first;
- only `layer4` and `fc` remain trainable.

In [ ]:
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# print the model architecture to verify the final layer before modification
print(model)

# replace the final layer to match the number of classes in CIFAR-10
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(class_names))

# Freeze all pretrained parameters so they are not updated during training.
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the parameters of layer4 so the last residual block can be fine-tuned.
for param in model.layer4.parameters():
    param.requires_grad = ... # TODO

# Unfreeze the parameters of the final fully connected layer.
for param in model.fc.parameters():
    param.requires_grad = ... # TODO

model = model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

### Training and evaluation utilities

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def fit_model(model, train_loader, val_loader, optimizer, criterion, device, epochs=2):
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    for epoch in range(epochs):
        start = time()

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        elapsed = time() - start
        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
            f"time={elapsed:.1f}s"
        )

    return history

### Fine-tune the model

Use:
- `CrossEntropyLoss`
- `Adam`
- learning rate `1e-4`
- `epochs = 2`

Then evaluate the model on the test subset.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

history = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=3,
)

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test loss: {test_loss:.4f}")
print(f"Test acc : {test_acc:.4f}")

### Visualize a few predictions

The next helper functions let us:
- undo normalization for display;
- compute class probabilities;
- show a few test images with predicted labels.

In [ ]:
@torch.no_grad()
def predict_proba_tensor(model, x):
    model.eval()
    logits = model(x.to(device))
    probs = torch.softmax(logits, dim=1)
    return probs.cpu()

def show_predictions(model, dataloader, class_names, n=8):
    images, labels = next(iter(dataloader))
    probs = predict_proba_tensor(model, images)
    preds = probs.argmax(dim=1)

    plt.figure(figsize=(14, 6))
    for i in range(n):
        plt.subplot(2, 4, i + 1)
        img = denormalize(images[i], imagenet_mean, imagenet_std).permute(1, 2, 0).clamp(0, 1)
        plt.imshow(img)
        title = f"True: {class_names[labels[i].item()]}\n Pred: {class_names[preds[i].item()]}"
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_predictions(model, test_loader, class_names, n=8)

---

## Part 4 — LIME explanation

### Reminder
**LIME** explains one prediction locally by:
1. perturbing the input image;
2. observing how the prediction changes;
3. fitting a simple interpretable model around the current sample;
4. identifying the most important image regions.

For images, LIME usually works with **superpixels** rather than single pixels.

### Step 1 — Select one correctly classified test image

Complete the helper below. It must return:
- the normalized tensor image,
- the true label index,
- the predicted label index.

In [ ]:
@torch.no_grad()
def get_correct_example(model, dataloader):
    model.eval()

    for images, labels in dataloader:
        logits = model(images.to(device))
        preds = logits.argmax(dim=1).cpu()

        for i in range(len(images)):
            if preds[i].item() == labels[i].item():
                return images[i], labels[i].item(), preds[i].item()

    raise RuntimeError("No correctly classified example found.")

image_tensor, true_label, pred_label = get_correct_example(model, test_loader)
print("True label     :", class_names[true_label])
print("Predicted label:", class_names[pred_label])

In [ ]:
def tensor_to_display_image(image_tensor):
    image = denormalize(image_tensor.cpu(), imagenet_mean, imagenet_std)
    image = image.permute(1, 2, 0).clamp(0, 1).numpy()
    return image

image_np = tensor_to_display_image(image_tensor)

plt.figure(figsize=(4, 4))
plt.imshow(image_np)
plt.title(f"True: {class_names[true_label]} | Pred: {class_names[pred_label]}")
plt.axis("off")
plt.show()

### Step 2 — Define a prediction function for LIME

LIME expects a function that:
- receives a **batch of images as NumPy arrays**;
- returns **class probabilities** as a NumPy array.

In the next cell, complete the function **without changing its name**.

What you need to do:
1. loop over the images in `images_np`;
2. convert each image from a NumPy array to a PIL image;
3. apply `test_transform` to each image;
4. stack all tensors into one batch;
5. run the model in evaluation mode / no-grad mode;
6. return the probabilities as a NumPy array.

Hints:
- `Image.fromarray(...)` expects pixel values in `uint8`, usually in `[0, 255]`;
- `torch.stack(tensors)` creates a mini-batch;
- use `torch.softmax(logits, dim=1)` to get probabilities;
- only fill the **right-hand side** of the assignments marked with `TODO`.


In [ ]:
def predict_for_lime(images_np):
    tensors = []

    for img in images_np:
        # TODO 1. Convert the image from [0, 1] float format to uint8 in [0, 255].
        img_uint8 = ...

        # TODO 2. Convert the NumPy image to a PIL image.
        pil_img = ...

        # TODO 3. Apply the test transform used by the model.
        tensor = ...
        tensors.append(tensor)

    # TODO 4. Build a batch and move it to the correct device.
    batch = ...

    with torch.no_grad():
        # TODO 5. Run the model on the batch.
        logits = ...

        # TODO 6. Convert logits to class probabilities.
        probs = ...

    # TODO 7. Return probabilities as a NumPy array on CPU.
    return ...


### Step 3 — Generate a LIME explanation

Use the `LimeImageExplainer` class to explain the prediction on `image_np`.

Read more on https://lime-ml.readthedocs.io/en/latest/lime.html#module-lime.lime_image

Instructions:
- create an explainer with `lime_image.LimeImageExplainer()`;
- call `explain_instance(...)` on `image_np`;
- use `classifier_fn=predict_for_lime`;
- use `top_labels=3` and `num_samples=1000`;
- then extract the explanation for the **predicted class** `pred_label`.

For `get_image_and_mask(...)`, use:
- `label=pred_label`
- `positive_only=True`
- `num_features=6`
- `hide_rest=False`

Hints:
- `explanation = explainer.explain_instance(...)`
- then `explanation.get_image_and_mask(...)`
- finally display the result with `mark_boundaries(...)`.

If `lime` is not available in your environment, run the next cell once.

In [ ]:
# Uncomment if needed:
# %pip install lime scikit-image

In [ ]:
from skimage.segmentation import mark_boundaries
from lime import lime_image

# TODO 1. Create the LIME image explainer.
explainer = ...

# TODO 2. Generate the explanation for image_np.
# Hint: use classifier_fn=predict_for_lime, top_labels=3, hide_color=0, num_samples=1000.
explanation = ...

# TODO 3. Extract the mask for the predicted class.
# Hint: the predicted class is stored in pred_label.
lime_image_vis, lime_mask = ...

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(image_np)
plt.title(f"Original image: {class_names[true_label]}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mark_boundaries(lime_image_vis / lime_image_vis.max(), lime_mask))
plt.title(f"LIME explanation for: {class_names[pred_label]}")
plt.axis("off")
plt.show()


### Questions on LIME

Answer briefly:
1. Does LIME highlight the object itself, the background, or both?
2. Why does LIME produce regions instead of a dense heatmap?
3. If you increase `num_samples`, what may happen to the quality and the runtime?

---

## Part 5 — Grad-CAM explanation

### Reminder
**Grad-CAM** uses:
- the activations of a convolutional layer;
- the gradients of the target class score with respect to these activations.

It produces a coarse heatmap showing **where** the CNN looked to support a prediction.

For ResNet18, a good choice is the last residual block:
```
model.layer4[-1]
```

### Step 1 — Register hooks on the target layer

We need to store:
- the forward activations;
- the backward gradients.

The following cell registers two hooks on the target layer for Grad-CAM.
- The forward hook is automatically called during the forward pass and stores the layer's output feature maps.
- The backward hook is automatically called during backpropagation and stores the gradients with respect to these feature maps.

This allows us to capture the information required to compute Grad-CAM without modifying the model architecture.

We use `model.layer4[-1]` because it is a deep convolutional layer that captures high-level semantic information while preserving spatial structure. The saved activations and gradients are later combined to generate the Grad-CAM heatmap.


In [ ]:
activations = None
gradients = None

def forward_hook(module, inputs, output):
    global activations
    activations = output.detach()

def backward_hook(module, grad_input, grad_output):
    global gradients
    gradients = grad_output[0].detach()

target_layer = model.layer4[-1]

forward_handle = target_layer.register_forward_hook(forward_hook)
backward_handle = target_layer.register_full_backward_hook(backward_hook)


### Step 2 — Compute the Grad-CAM heatmap

Complete the next function step by step.

Main steps:
1. switch the model to evaluation mode;
2. clear previously stored activations / gradients;
3. run a forward pass on one image;
4. choose the target class (use the predicted class if `class_idx is None`);
5. backpropagate the score of that class;
6. average gradients spatially to get one weight per channel;
7. compute the weighted sum of the activation maps;
8. apply ReLU;
9. normalize the heatmap.

Hints:
- add a batch dimension with `unsqueeze(0)`;
- `score = logits[:, class_idx].sum()` is enough for one image;
- average the gradients with `mean(dim=(2, 3), keepdim=True)`;
- combine weights and activations, then sum over the channel dimension.


In [ ]:
def compute_gradcam(model, image_tensor, class_idx=None):
    global activations, gradients

    # TODO 1. Put the model in evaluation mode.
    ...

    # TODO 2. Reset stored activations and gradients.
    activations = ...
    gradients = ...

    # TODO 3. Add a batch dimension and move the image to the device.
    x = ...

    # TODO 4. Forward pass.
    logits = ...

    # TODO 5. If no class is provided, use the predicted class.
    if class_idx is None:
        class_idx = ...

    # TODO 6. Select the score of the target class.
    score = ...

    # TODO 7. Backpropagate this score.
    model.zero_grad()
    ...

    # TODO 8. Average gradients over spatial dimensions to get channel weights.
    pooled_gradients = ...

    # TODO 9. Weighted sum of activation maps over channels.
    cam = ...

    # TODO 10. Keep only positive contributions.
    cam = ...

    # Optional: cam -= cam.min()

    # TODO 11. Normalize the heatmap to [0, 1].
    cam = ...

    # TODO 12. Return the heatmap as a NumPy array, together with the class index.
    return ...


### Step 3 — Overlay the heatmap on the original image

Now visualize the Grad-CAM result.

1. call `compute_gradcam(...)` on `image_tensor`;
2. convert the returned CAM to a tensor of shape `(1, 1, H, W)`;
3. resize it to `(224, 224)` with `F.interpolate(...)`;
4. display the original image `image_np`;
5. overlay the heatmap with `plt.imshow(..., alpha=0.4)`.

Hints:
- use two `unsqueeze(...)` calls to create shape `(1, 1, H, W)`;
- after interpolation, use `.squeeze().numpy()`;
- use `cmap="jet"` for the heatmap.


In [ ]:
import torch.nn.functional as F

cam, cam_class_idx = compute_gradcam(model, image_tensor, class_idx=pred_label)

cam_tensor = torch.tensor(cam).unsqueeze(0).unsqueeze(0)
cam_resized = F.interpolate(cam_tensor, size=(224, 224), mode="bilinear", align_corners=False)
cam_resized = cam_resized.squeeze().numpy()

plt.figure(figsize=(5, 5))
plt.imshow(image_np)
plt.imshow(cam_resized, cmap="jet", alpha=0.4)
plt.title(f"Grad-CAM for: {class_names[cam_class_idx]}")
plt.axis("off")
plt.show()

### Questions on Grad-CAM

Answer briefly:
1. Does Grad-CAM focus on the same zones as LIME?
2. Why is the Grad-CAM map usually coarse?
3. Why do we typically apply Grad-CAM to a late convolutional layer rather than to the classifier?

---

## Part 6 — Compare LIME and Grad-CAM

Write a short comparison based on your outputs.

Suggested points:
1. Which method is more intuitive to read?
2. Which method seems more local and region-based?
3. Which method seems more directly tied to the internal CNN architecture?
4. In practice, when might you prefer one over the other?

## End of lab